In [ ]:
answers = {'csrfmiddlewaretoken': ['<csrf-token>'], 'responses': ['{"question_1":{"forminput":"1873"},"question_2":{"forminput":"Cello"},"question_3":{"forminput":"The Civil War"},"question_4":{"forminput":"1905"},"question_5":{"forminput":"Piano Concerto No. 2"},"question_6":{"forminput":"Lack of harmony"},"question_7":{"forminput":"His music is technical but emotionally disconnected."},"question_8":{"forminput":"He performed his own compositions."},"question_9":{"highlights":{"yellow":[{"id":1,"text":"e displayed prodigious talent, which was nurtured by a family deeply involved in the arts. His mother, a talented pianist, recognized "}],"pink":[]}},"question_10":{"forminput":"A sonata"},"question_11":{"forminput":"Anton Arensky and Sergey Taneyev"},"question_12":{"forminput":"Composing new music"},"question_13":{"forminput":"He loved to play piano"},"question_14":{"forminput":"Rachmaninoff was not well received in his time."},"question_15":{}}'], 'submit_quiz': ['1']}

In [ ]:
import json 
from pymongo import MongoClient
from pymongo.server_api import ServerApi
practice_id = "<practice-id>"
PROD = False
uri = "" if PROD else  ""
client = MongoClient(uri, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
MDB = client.reading
quiz = MDB.questions.find_one({"practice_id": practice_id})



In [ ]:
import json
from copy import deepcopy

def grade_quiz(questions, user_answers):
    # Create a deep copy of the questions dictionary to avoid modifying the original
    _questions = deepcopy(questions)
    
    # Parse user answers
    user_answers = json.loads(user_answers['responses'][0])
    
    for question in _questions['quiz']['questions']:
        q_num = question['question_num']
        q_type = question['type']
        user_answer = user_answers.get(f'question_{q_num}', {}).get('forminput')
        
        if q_type == 'multiple':
            correct_index = question['correct_index']
            correct_answer = question['answers'][int(correct_index)]
            is_correct = user_answer == correct_answer
            user_answer_index = question['answers'].index(user_answer) if user_answer in question['answers'] else None
            
            # Add grading information directly to the question dictionary
            question['user_answer'] = user_answer
            question['user_answer_index'] = user_answer_index
            question['is_correct'] = is_correct
        
        elif q_type == 'highlight':
            correct_highlight = question['correct_highlight'].strip()
            user_highlights = user_answers.get(f'question_{q_num}', {}).get('highlights', {}).get('yellow', [])
            highlighted_text = ' '.join([highlight['text'] for highlight in user_highlights]).strip()
            correct_chars = set(correct_highlight)
            highlighted_chars = set(highlighted_text)
            correct_chars_highlighted = len(correct_chars.intersection(highlighted_chars))
            total_correct_chars = len(correct_chars)
            grade = (correct_chars_highlighted / total_correct_chars) * 100 if total_correct_chars > 0 else 0           
            question['user_answer'] = highlighted_text
            question['grade'] = round(grade, 2)  # Round to 2 decimal places
            question['feedback'] = [{"type": "AI","message" : "This is the AI feedback for the user's answer."}]
        
        elif q_type == 'open':
            # Add user answer directly to the question dictionary
            question['user_answer'] = user_answer
            question['feedback'] = [{"type": "AI","message" : "This is the AI feedback for the user's answer."}]
            question['grade'] = 100
    _questions[''] = True
    return _questions

In [ ]:
results = grade_quiz(quiz, answers)

In [ ]:
results

In [ ]:
MDB.questions.update_one({"practice_id": practice_id}, {"$set": results})

In [ ]:
def pedro(life):
  return "Success"